In [74]:
import pubchempy as pcp
import reframed
import pandas as pd
import numpy as np

# Get inchikeys
The purpose of this notebook is to get inchikeys for all metabolites in order to use classyfire to get the metabolite class

In [42]:
model_fn = '../../data/5_FBA/universe_bacteria.xml.gz'
model = reframed.load_cbmodel(model_fn)

In [43]:
met_to_inchi = {}
added_mets = []
abbrv_to_name = {}
data = []
for m_id in model.metabolites:
    m = model.metabolites[m_id]
    abbrv = m_id.lstrip('M_')[:-2]
    try:
        inchi = m.metadata['inchikey'][0]
    except KeyError:
        inchi = None

    if inchi is not None:
        met_to_inchi[abbrv] = inchi
    else:
        try:
            prev_inchi = met_to_inchi[abbrv]
        except KeyError:
            prev_inchi = None
        if prev_inchi is None:
            met_to_inchi[abbrv] = None
    
    if not abbrv in added_mets:
        added_mets.append(abbrv)
        abbrv_to_name[abbrv] = m.name
        data.append([abbrv, m.name])

df = pd.DataFrame(data, columns=['abbrv', 'name'])
df['InChI'] = df['abbrv'].map(met_to_inchi)

In [44]:
inchi_fn = '../../data/5_FBA/inchi_keys.csv'
df.to_csv(inchi_fn, sep = '\t', index=False)

In [45]:
met_to_inchi
inchi_to_met = {val: key for key, val in met_to_inchi.items() if val is not None}

In [46]:

inchis = list(set(df.loc[df.InChI.notnull(),  'InChI'].values))

In [47]:
# pd.Series(inchis).iloc[:800].to_csv('../../data/5_FBA/inchi_keys_series_1.tsv', sep = '\t', index=False)
# pd.Series(inchis).iloc[800:].to_csv('../../data/5_FBA/inchi_keys_series_2.tsv', sep = '\t', index=False)

In [48]:
compounds = pcp.get_compounds(inchis, 'inchikey')

In [49]:
completed_inchis = []
pubchem_data = []
for c in compounds:
    completed_inchis.append(c.inchikey)
    pubchem_data.append([c.inchikey, c.cid, c.iupac_name, c.molecular_formula, c.molecular_weight, c.isomeric_smiles, c.canonical_smiles])



In [50]:
for i in inchis:
    if i not in completed_inchis:
        abbrv = inchi_to_met[i]
        mname = abbrv_to_name[abbrv]
        try:
            m = model.metabolites[f'M_{abbrv}_c']
        except KeyError:
            try:
                m = model.metabolites[f'M_{abbrv}_p']
            except KeyError:
                try:
                    m = model.metabolites[f'M_{abbrv}_e']
                except KeyError:
                    print(f'No metabolite found for {abbrv}')
                    continue
        try:
            charge = float(m.metadata['CHARGE'])
        except KeyError:
            print(f'No charge found for {abbrv}')
            charge = None
        
        compounds_i = pcp.get_compounds(mname, 'name')
        
        if not len(compounds_i):
            print(f'No compound found for {mname}')
            continue
        else:    
            c = compounds[0]
            completed_inchis.append(c.inchikey)
            print(mname, c.iupac_name)
            met_to_inchi[abbrv] = c.inchikey
            pubchem_data.append([c.inchikey, c.cid, c.iupac_name, c.molecular_formula, c.molecular_weight, c.isomeric_smiles, c.canonical_smiles, charge])

    

No compound found for Two linked disacharide pentapeptide murein units (uncrosslinked, middle of chain)
No compound found for Phosphatidylglycerophosphate (dihexadec-9-enoyl, n-C16:1)
Quinate 5-[2-[3-[[(2R)-4-[[[(2R,3S,4R,5R)-5-(6-aminopurin-9-yl)-4-hydroxy-3-phosphonatooxyoxolan-2-yl]methoxy-oxidophosphoryl]oxy-oxidophosphoryl]oxy-2-hydroxy-3,3-dimethylbutanoyl]amino]propanoylamino]ethylsulfanyl]-3-hydroxy-3-methyl-5-oxopentanoate
2-Acyl-sn-glycero-3-phosphoethanolamine (n-C12:0) 5-[2-[3-[[(2R)-4-[[[(2R,3S,4R,5R)-5-(6-aminopurin-9-yl)-4-hydroxy-3-phosphonatooxyoxolan-2-yl]methoxy-oxidophosphoryl]oxy-oxidophosphoryl]oxy-2-hydroxy-3,3-dimethylbutanoyl]amino]propanoylamino]ethylsulfanyl]-3-hydroxy-3-methyl-5-oxopentanoate
2-Hydroxy-5-methyl-cis,cis-muconate 5-[2-[3-[[(2R)-4-[[[(2R,3S,4R,5R)-5-(6-aminopurin-9-yl)-4-hydroxy-3-phosphonatooxyoxolan-2-yl]methoxy-oxidophosphoryl]oxy-oxidophosphoryl]oxy-2-hydroxy-3,3-dimethylbutanoyl]amino]propanoylamino]ethylsulfanyl]-3-hydroxy-3-methyl-5-oxop

In [51]:
inchi_to_met = {val: key for key, val in met_to_inchi.items() if val is not None}

In [52]:
pubchem_df = pd.DataFrame(pubchem_data, columns=['InChI', 'CID', 'IUPAC_name', 'Molecular_formula', 'Molecular_weight', 'Isomeric_smiles', 'Canonical_smiles', 'Charge'])
pubchem_df['abbrv'] = pubchem_df['InChI'].map(inchi_to_met)
pubchem_df['Name'] = pubchem_df['abbrv'].map(abbrv_to_name)

In [81]:
m = model.metabolites['M_mhpglu_c']
m.metadata

OrderedDict([('FORMULA', 'C30H35N9O12'),
             ('SBOTerm', 'SBO:0000247'),
             ('kegg.compound', ['C04489']),
             ('chebi',
              ['CHEBI:12147',
               'CHEBI:17614',
               'CHEBI:20613',
               'CHEBI:2099',
               'CHEBI:58207']),
             ('hmdb', ['HMDB12177']),
             ('inchikey', ['HVRNKDVLFAVCJF-VJANTYMQSA-J']),
             ('biocyc', ['META:CPD-1302']),
             ('metanetx.chemical', ['MNXM2623']),
             ('seed.compound', ['cpd02738'])])

In [75]:
# Add charge to missing metabolites
for i, row in pubchem_df.iterrows():
    if np.isnan(row['Charge']):
        abbrv = row['abbrv']
        try:
            m = model.metabolites[f'M_{abbrv}_c']
        except KeyError:
            try:
                m = model.metabolites[f'M_{abbrv}_p']
            except KeyError:
                try:
                    m = model.metabolites[f'M_{abbrv}_e']
                except KeyError:
                    print(f'No metabolite found for {abbrv}')
                    continue
        try:
            charge = float(m.metadata['CHARGE'])
        except KeyError:
            print(f'No charge found for {abbrv}')
            charge = None
        pubchem_df.at[i, 'Charge'] = charge

No charge found for mhpglu
No charge found for 3opcoa
No charge found for pt2coa
No charge found for lac6p
No charge found for cspmd
No charge found for 3oxdhscoa
No charge found for 24dhhed
No charge found for 2hmc
No charge found for 5cmhm
No charge found for salc
No charge found for cchoxod
No charge found for 4cml
No charge found for d5kgp
No charge found for 23dhacoa
No charge found for 2oxpaccoa
No charge found for uaaGgla
No charge found for uaGgla
No charge found for s17bp
No charge found for 2sephchc
No charge found for hkmpp
No charge found for hibcoa
No charge found for ptcoa
No charge found for benzcoa
No charge found for trans_dd2coa
No charge found for 3hbycoa
No charge found for shcl
No charge found for lipoamp
No charge found for hepdp
No charge found for achms
No charge found for cellb6p
No charge found for dccoa
No charge found for pendp
No charge found for dkdi
No charge found for 4h2kpi
No charge found for mucl
No charge found for g6p_A
No charge found for dh3mchc
N

In [84]:
# pubchem_df.Charge.isna().sum()

In [53]:
# pubchem_df.to_csv('../../data/5_FBA/pubchem_data.tsv', sep = '\t', index=False)

In [54]:
# inchi_only_df = pubchem_df[['Name', 'InChI']]
# inchi_only_df.to_csv('../../data/5_FBA/inchi_only.tsv', sep = '\t', index=False)

In [85]:
smiles_inly_df = pubchem_df[['Name', 'Canonical_smiles']]
# remove duplicated rows based on Name
smiles_inly_df = smiles_inly_df.drop_duplicates(subset=['Name'])
smiles_inly_df.to_csv('../../data/5_FBA/smiles_only_1.tsv', sep = '\t', index=False)
smiles_inly_df.shape


(971, 2)

# Now parse the classify file

In [86]:
classy_fire = pd.read_csv('../../data/5_FBA/classy_fire.csv')

In [87]:
classy_fire = classy_fire.loc[classy_fire.ChemOntID != 'NotAvailable']

In [88]:
idx = classy_fire['ClassifiedResults'].str.contains('alternative_parents:').astype(bool)
classy_fire = classy_fire.loc[~idx]

In [89]:
idx = classy_fire['ClassifiedResults'].str.contains('Intermediate_nodes:').astype(bool)
classy_fire = classy_fire.loc[~idx]

In [90]:
classy_fire.ClassifiedResults.str.split(':')

0                           [Kingdom,  Organic compounds]
1          [Superclass,  Lipids and lipid-like molecules]
2                                   [Class,  Fatty Acyls]
3                      [Subclass,  Fatty acyl thioesters]
6        [Direct_parent,  3-hydroxy-3-alkylglutaryl CoAs]
                               ...                       
23009                       [Kingdom,  Organic compounds]
23010                           [Superclass,  Benzenoids]
23011       [Class,  Benzene and substituted derivatives]
23012                        [Subclass,  Benzyl alcohols]
23013                   [Direct_parent,  Benzyl alcohols]
Name: ClassifiedResults, Length: 4598, dtype: object

In [91]:
classy_fire.ClassifiedResults.str.contains('parent')

0        False
1        False
2        False
3        False
6         True
         ...  
23009    False
23010    False
23011    False
23012    False
23013     True
Name: ClassifiedResults, Length: 4598, dtype: bool

In [92]:
# Assuming 'ClassifiedResults' is the column you want to split
classy_fire[['level', 'value']] = classy_fire.ClassifiedResults.str.split(':', expand=True)
classy_fire['value'] = classy_fire['value'].str.lstrip(' ')

In [93]:
# Pivot the DataFrame to make each level a column
wide_df = classy_fire.pivot(index='CompoundID', columns='level', values='value').reset_index()


In [94]:
wide_df.head()

level,CompoundID,Class,Direct_parent,Kingdom,Subclass,Superclass
0,R 2 Oxo 3 methylpentanoate C6H9O3,Keto acids and derivatives,Short-chain keto acids and derivatives,Organic compounds,Short-chain keto acids and derivatives,Organic acids and derivatives
1,R 5 Diphosphomevalonate C6H10O10P2,Organic oxoanionic compounds,Organic pyrophosphates,Organic compounds,Organic pyrophosphates,Organic oxygen compounds
2,R 5 Phosphomevalonate C6H10O7P,Hydroxy acids and derivatives,Short-chain hydroxy acids and derivatives,Organic compounds,Short-chain hydroxy acids and derivatives,Organic acids and derivatives
3,R Acetoin C4H8O2,Organooxygen compounds,Acyloins,Organic compounds,Carbonyl compounds,Organic oxygen compounds
4,R Mevalonate C6H11O4,Fatty Acyls,Hydroxy fatty acids,Organic compounds,Fatty acids and conjugates,Lipids and lipid-like molecules


In [95]:
# Make a simpified classification
for i, row in wide_df.iterrows():
    if row['Class'] == 'Organooxygen compounds':
        defined_class = 'Organooxygen compounds'
    elif row['Class'] == 'Carboxylic acids and derivatives':
        if row['Subclass'] == "Amino acids, peptides, and analogues":
            defined_class = 'Amino acids'
        else:
            defined_class = 'Carboxylic acids'
    elif row['Class'] in ['Hydroxy acids and derivatives','Keto acids and derivatives']:
        defined_class = 'Keto / hydroxy acids'
    else:
        defined_class = 'Other'
    wide_df.at[i, 'Defined class'] = defined_class


In [96]:
wide_df.head(40)

level,CompoundID,Class,Direct_parent,Kingdom,Subclass,Superclass,Defined class
0,R 2 Oxo 3 methylpentanoate C6H9O3,Keto acids and derivatives,Short-chain keto acids and derivatives,Organic compounds,Short-chain keto acids and derivatives,Organic acids and derivatives,Keto / hydroxy acids
1,R 5 Diphosphomevalonate C6H10O10P2,Organic oxoanionic compounds,Organic pyrophosphates,Organic compounds,Organic pyrophosphates,Organic oxygen compounds,Other
2,R 5 Phosphomevalonate C6H10O7P,Hydroxy acids and derivatives,Short-chain hydroxy acids and derivatives,Organic compounds,Short-chain hydroxy acids and derivatives,Organic acids and derivatives,Keto / hydroxy acids
3,R Acetoin C4H8O2,Organooxygen compounds,Acyloins,Organic compounds,Carbonyl compounds,Organic oxygen compounds,Organooxygen compounds
4,R Mevalonate C6H11O4,Fatty Acyls,Hydroxy fatty acids,Organic compounds,Fatty acids and conjugates,Lipids and lipid-like molecules,Other
5,S 3 Hydroxybutyryl CoA C25H38N7O18P3S,Fatty Acyls,3-hydroxyacyl CoAs,Organic compounds,Fatty acyl thioesters,Lipids and lipid-like molecules,Other
6,S 3 Hydroxyisobutyryl CoA C25H38N7O18P3S,Fatty Acyls,Acyl CoAs,Organic compounds,Fatty acyl thioesters,Lipids and lipid-like molecules,Other
7,(-)-Ureidoglycolate,Carboxylic acids and derivatives,N-carbamoyl-alpha amino acids,Organic compounds,"Amino acids, peptides, and analogues",Organic acids and derivatives,Amino acids
8,"(2,3-Dihydroxybenzoyl)adenylate",Purine nucleotides,Purine ribonucleoside monophosphates,Organic compounds,Purine ribonucleotides,"Nucleosides, nucleotides, and analogues",Other
9,"(2R,4S)-2-methyl-2,3,3,4-tetrahydroxytetrahydr...",Organooxygen compounds,Pentoses,Organic compounds,Carbohydrates and carbohydrate conjugates,Organic oxygen compounds,Organooxygen compounds


In [97]:
full_df = pubchem_df.merge(wide_df, left_on='Name', right_on='CompoundID', how='left')

In [98]:
name_to_abbrv = {val: key for key, val in abbrv_to_name.items()}

In [99]:
full_df['Metabolite ID'] = full_df['Name'].map(name_to_abbrv)
# Move the 'Name' column to the first position
columns = ['Metabolite ID', 'Name'] + [col for col in full_df.columns if not col in ['Metabolite ID', 'Name']]
full_df = full_df[columns]


In [100]:
full_df.to_csv('../../data/5_FBA/carveme_universe_met_info_and_class.csv')